# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible via the following URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

This dataset details the analysis of protein abundance, modifications, and sequences in human samples. It contains multiple record sets describing mass spectrometry results extracted from extracellular vesicles.

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset's metadata and access its records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via the mlcroissant Dataset helper
dataset = mlc.Dataset(url)

# Print the top-level dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Explore the available record sets in the dataset and their associated field `@id` values.

**Note:** In mlcroissant, record sets and fields are referenced by their `@id` according to the Croissant schema.

In [ ]:
# List all available record sets and their IDs in the dataset
record_set_ids = [r['@id'] for r in dataset.metadata.record_sets]
print("Available record sets (by @id):")
for rid in record_set_ids:
    print(f"  - {rid}")

# List fields for each record set, referencing their @id
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):  # single field
        fields = [fields]
    for field in fields:
        # Each field is a reference; resolve by @id
        fid = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"    Field: {fid}")

## 3. Data Extraction
Extract records from a record set and load them into a DataFrame for exploration.

For demonstration, we will load the "main" record set, which contains the protein information. For all references, we will use the actual `@id` from the Croissant schema.

In [ ]:
# Let's choose the main protein data record set for extraction
# Find the record set labeled with 'main' or the one that is most likely to contain measurements
main_record_set_id = None
for r in dataset.metadata.record_sets:
    name = r.get('name', '').lower() if 'name' in r else ''
    if 'protein' in name or 'main' in name or 'results' in name:
        main_record_set_id = r['@id']
        break
if main_record_set_id is None:
    # If there is only one, use that one
    main_record_set_id = record_set_ids[0]

print(f"Using main record set: {main_record_set_id}")

record_sets_to_load = [main_record_set_id]
dataframes = {}
for rsid in record_sets_to_load:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

print(f"Fields in record set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will now explore the main record set using typical processing such as filtering, normalization, and grouping. All fields must be referenced by their `@id` (as in the schema overview above).

- We will select a numeric field (e.g., the peptide count or molecular weight field) and apply threshold-based filtering and normalization.
- If a categorical field (e.g., protein type or description) is available, group and summarize by it.

In [ ]:
# Determine appropriate fields by their @id
# Let's find numeric fields (like coverage, peptide_count, or molecular_weight)
df = dataframes[main_record_set_id]

# For demonstration: check typical numeric columns
numeric_candidates = [col for col in df.columns if 'weight' in col or 'count' in col or 'coverage' in col or df[col].dtype in ['int64', 'float64']]
print("Numeric field candidates:", numeric_candidates)

# Select a default numeric field (fall back if not present)
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # Fall back to first numerical column
    numeric_field_id = df.select_dtypes(include=['number']).columns[0]

print(f"Analyzing field: {numeric_field_id}")

# Filter records (e.g., proteins with molecular weight > threshold)
threshold = df[numeric_field_id].mean() # If you want, use e.g., 10 or df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Pick a group field (e.g., a categorical field such as description)
group_field_candidates = [col for col in df.columns if 'description' in col or 'type' in col or df[col].dtype == object]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    print(f"\nGrouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped summary by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the data, using the selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric_field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Optional: Boxplot by group field
if 'group_field_id' in locals():
    plt.figure(figsize=(12, 4))
    sns.boxplot(y=df[numeric_field_id], x=df[group_field_id])
    plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have explored the FAIR^2 dataset schema, loaded mass spectrometry protein records by their record set `@id`, examined main numeric and categorical features, performed filtering and normalization, and visualized distributions.

- The dataset's record sets and fields are referenced by their unique `@id` in all steps, ensuring consistent access and reproducible analysis.
- mlcroissant enables easy inspection, extraction, and transformation of rich multi-table datasets, fully leveraging the Croissant schema's structure.

For in-depth analysis, experiment with additional fields and combinations, and refer to the [mlcroissant documentation](https://croissant.mlcommons.org/).